# ML Model Serving & Deployment Cheatsheet

> Patterns and frameworks for serving ML models in production.

## FastAPI Model Server

In [ ]:
# FastAPI Model Server (reference code)
from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np

app = FastAPI(title="ML Model API")

class PredictionRequest(BaseModel):
    features: list[float]

class PredictionResponse(BaseModel):
    prediction: float
    confidence: float

# model = load_model("model.pkl")

@app.post("/predict", response_model=PredictionResponse)
async def predict(request: PredictionRequest):
    features = np.array(request.features).reshape(1, -1)
    # prediction = model.predict(features)[0]
    # confidence = float(max(model.predict_proba(features)[0]))
    prediction = 0.95  # placeholder
    confidence = 0.87  # placeholder
    return PredictionResponse(prediction=prediction, confidence=confidence)

@app.get("/health")
async def health():
    return {"status": "healthy"}

# Run: uvicorn app:app --host 0.0.0.0 --port 8000 --workers 4
print("FastAPI model server pattern defined")

## TorchServe

```bash
# Package model
torch-model-archiver --model-name my_model --version 1.0 \
  --model-file model.py --serialized-file model.pth \
  --handler text_classifier --export-path model_store/

# Start server
torchserve --start --model-store model_store --models my_model=my_model.mar --ncs

# Inference
curl http://localhost:8080/predictions/my_model -T input.txt

# Management API
curl http://localhost:8081/models                       # List models
curl http://localhost:8081/models/my_model              # Model details
curl -X PUT 'http://localhost:8081/models/my_model?min_worker=2&max_worker=4'  # Scale
curl -X DELETE http://localhost:8081/models/my_model    # Remove model

# Stop server
torchserve --stop
```

## TensorFlow Serving

```bash
# Save TF model in SavedModel format
# model.save("saved_model/my_model/1/")

# Run with Docker
docker run -p 8501:8501 \
  -v "$(pwd)/saved_model/my_model:/models/my_model" \
  -e MODEL_NAME=my_model \
  tensorflow/serving

# REST API prediction
curl -X POST http://localhost:8501/v1/models/my_model:predict \
  -H "Content-Type: application/json" \
  -d '{"instances": [[1.0, 2.0, 3.0, 4.0]]}'

# Check model status
curl http://localhost:8501/v1/models/my_model
```

## NVIDIA Triton Inference Server

```bash
# Model repository structure
# model_repo/
#   my_model/
#     config.pbtxt
#     1/
#       model.onnx

# Run with Docker
docker run --gpus all -p 8000:8000 -p 8001:8001 -p 8002:8002 \
  -v $(pwd)/model_repo:/models \
  nvcr.io/nvidia/tritonserver:24.05-py3 \
  tritonserver --model-repository=/models

# Health check
curl http://localhost:8000/v2/health/ready

# List models
curl http://localhost:8000/v2/models

# HTTP inference
curl -X POST http://localhost:8000/v2/models/my_model/infer \
  -H "Content-Type: application/json" \
  -d '{"inputs":[{"name":"input","shape":[1,4],"datatype":"FP32","data":[[1.0,2.0,3.0,4.0]]}]}'
```

## BentoML

In [ ]:
# BentoML Pattern (reference code)
import bentoml

# Save model to BentoML model store
# bentoml.sklearn.save_model("fraud_detector", trained_model)

# Define service
# @bentoml.service(resources={"gpu": 1, "memory": "4Gi"})
# class FraudDetector:
#     model = bentoml.models.get("fraud_detector:latest")
#
#     @bentoml.api
#     def predict(self, features: list[float]) -> dict:
#         result = self.model.predict([features])
#         return {"prediction": int(result[0])}

# CLI:
# bentoml serve service:FraudDetector    # Start dev server
# bentoml build                          # Build Bento
# bentoml containerize fraud_detector:latest  # Create Docker image

print("BentoML service pattern defined")

## ONNX Export & Optimization

In [ ]:
# ONNX Export & Optimization (reference code)
import torch

# Export PyTorch model to ONNX
# dummy_input = torch.randn(1, 3, 224, 224)
# torch.onnx.export(
#     model, dummy_input, "model.onnx",
#     input_names=["input"],
#     output_names=["output"],
#     dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
# )

# Optimize ONNX model
import onnxruntime as ort

# Quantize (reduce model size, speed up inference)
from onnxruntime.quantization import quantize_dynamic, QuantType

# quantize_dynamic("model.onnx", "model_quantized.onnx", weight_type=QuantType.QInt8)

# Run inference with ONNX Runtime
# session = ort.InferenceSession("model_quantized.onnx", providers=["CUDAExecutionProvider"])
# result = session.run(None, {"input": input_data})

print("ONNX export and optimization patterns defined")

## Model Quantization

In [ ]:
# PyTorch Quantization (reference code)
import torch

# Dynamic Quantization (simplest)
# quantized_model = torch.quantization.quantize_dynamic(
#     model, {torch.nn.Linear}, dtype=torch.qint8
# )

# Static Quantization (better accuracy)
# model.eval()
# model.qconfig = torch.quantization.get_default_qconfig("x86")
# prepared_model = torch.quantization.prepare(model)
# # Run calibration with representative data
# for batch in calibration_loader:
#     prepared_model(batch)
# quantized_model = torch.quantization.convert(prepared_model)

print("Quantization patterns: dynamic (easy, all models) and static (better, needs calibration)")
print("Benefits: ~2-4x smaller model, ~2-3x faster inference, lower memory")

## Deployment Patterns

### Blue-Green Deployment

```yaml
# blue-green-deployment.yaml
apiVersion: v1
kind: Service
metadata:
  name: model-service
spec:
  selector:
    app: model-server
    version: green        # Switch between blue/green
  ports:
    - port: 80
      targetPort: 8000
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: model-server-green
spec:
  replicas: 3
  selector:
    matchLabels:
      app: model-server
      version: green
  template:
    metadata:
      labels:
        app: model-server
        version: green
    spec:
      containers:
        - name: model
          image: model-server:v2
          ports:
            - containerPort: 8000
```

### Canary Deployment

```yaml
# Istio-based canary
apiVersion: networking.istio.io/v1beta1
kind: VirtualService
metadata:
  name: model-service
spec:
  http:
    - route:
        - destination:
            host: model-server
            subset: stable
          weight: 90
        - destination:
            host: model-server
            subset: canary
          weight: 10            # 10% traffic to new version
```

## Load Testing

In [ ]:
# Locust Load Testing for ML Endpoints (reference code)
from locust import HttpUser, task, between

class MLModelUser(HttpUser):
    wait_time = between(0.5, 2)

    @task(3)
    def predict(self):
        self.client.post("/predict", json={
            "features": [1.0, 2.0, 3.0, 4.0]
        })

    @task(1)
    def health_check(self):
        self.client.get("/health")

# Run: locust -f locustfile.py --host http://localhost:8000
print("Locust load test pattern defined")
print("Key metrics to watch: p50/p95/p99 latency, requests/sec, error rate")

## Framework Comparison

| Framework | Best For | GPU | Multi-Model | Protocol |
|-----------|----------|-----|-------------|----------|
| **FastAPI** | Custom APIs, simple models | Manual | Manual | REST |
| **TorchServe** | PyTorch models | Yes | Yes | REST/gRPC |
| **TF Serving** | TensorFlow models | Yes | Yes | REST/gRPC |
| **Triton** | Multi-framework, high perf | Yes | Yes | REST/gRPC |
| **BentoML** | Easy packaging, any framework | Yes | Yes | REST/gRPC |

## Interview Scenarios

**Q: How do you deploy ML models with zero downtime?**
> Use blue-green deployment: deploy the new model alongside the old one, run validation tests against the green deployment, then switch traffic. If issues arise, instantly revert by switching traffic back to blue. For gradual rollout, use canary deployment with progressive traffic shifting.

**Q: How do you reduce model inference latency?**
> Progressive optimization: (1) model quantization (INT8/FP16), (2) ONNX Runtime conversion, (3) request batching, (4) model distillation, (5) GPU inference with Triton, (6) caching frequent predictions, (7) edge deployment for latency-critical apps.

**Q: How do you choose a model serving framework?**
> Consider: (1) model framework (TorchServe for PyTorch, TF Serving for TF, Triton for multi-framework), (2) performance requirements (Triton for highest throughput), (3) ease of use (BentoML/FastAPI for simpler needs), (4) infrastructure (cloud-managed endpoints vs self-hosted).